# Chapter 1

### Pytorch basics

```
some_list = [[1, 2, 3], [4, 5, 6]]
np_array = np.array(some_list)
np_tensor = torch.from_numpy(np_array) # creating tensors from array
import torch
tensor = torch.tensor(some_list) # Creating tensor from list, data type is based on the input value type
tensor2 = torch.Tensor(some_list) # Creating tensor from list with float32 as default data type
tensor.dtype # See data type of tensors
tensor.device # See what device the tensor is using
a + b # Tensor addition (subtraction, multiplication are done same way as elementwise operation)

torch.manual_seed(seed) # Setting seed 
torch.rand_like(some_array, dtype=torch.float) # Create a tensor with random numbers with same dimension as given array
torch.ones_like(some_array, dtype=torch.float) # Create a tensor of 1s with same dimension as given array
shape = (2,3,)
rand_tensor = torch.rand(shape) # Create a tensor of random numbers with specified shape
ones_tensor = torch.ones(shape) # Create a tensor of 1s with specified shape
zeros_tensor = torch.zeros(shape) # Create a tensor of 0s with specified shape
tensor.shape # Shape of tensor
tensor.dtype # Datatype of tensor
tensor.device  # Device tensor is stored on
some_tensor = torch.arange(12).reshape(3, 4).float() # Creating, reshaping and giving data type to a tensor
print('First row: ',some_tensor[0])
print('First column: ', some_tensor[:, 0])
print('Last column:', some_tensor[:, -1])
t1 = torch.cat([tensor1, tensor2, tensor3], dim=1) # Horizontal concatenation, increasing features/columns
matrix_mul = tensor @ tensor.T # matrix multiplication
matrix_mul = tensor.matmul(tensor.T)  # matrix multiplication with a tensor and the TRANSPOSE of it
matrix_mul = torch.mm(mat.t(), mat) # matrix multiplication with a tensor and the TRANSPOSE of it
normal_mul = tensor1 * tensor2
normal_mul = tensor1.mul(tensor2)
agg = tensor.sum() # summing
agg_item = agg.item()  # extracting value from a tensor

x = torch.arange(4.0)
x.requires_grad_(True)  # Same as `x = torch.arange(4.0, requires_grad=True)` which means we can do derivation with respect to it
y = 2 * torch.dot(x, x) # function y = x*x. square of each x and then sum for dot product: 2⋅(0+1+4+9)=2⋅14=28
y.backward() # Derivative of y which is (2.2.x = 4.x) : 4(0, 1, 2, 3) = (0, 4, 8, 12)
x.grad # Show derivated result

x = np.linspace(-np.pi, np.pi, 100)
x = torch.tensor(x, requires_grad=True)
y = torch.sin(x)
y.backward(torch.ones_like(x)) # specify  initial gradients if x is not scalar when back-propagation
# NOTE : y x is not a scalar. so `y.backward()` will throw error. any y using such x which is not a scalar will require initial grads
# raise NotImplementedError
x.grad # Show derivated result

def sigmoid(x): # turns a value from 0 to 1
    return 1/(1+ torch.exp(-x))

def softmax(X): # turns a row of values into probabilities. sum of the row is 1
    result = torch.zeros_like(X) # This is where we will store the probability values
    for i in range(X.shape[0]): # iterate over each row
        row = X[i]
        max_val = torch.max(row) # get the maximum value of the row
        exp_row = torch.exp(row - max_val) # scale each value of row so that maximum value of row is 0
        row_sum = torch.sum(exp_row) # get the sum of all values in the row
        softmax_row = exp_row / row_sum # get probability for each value
        result[i] = softmax_row # store the result row 
    return result

def linear(X, W, b): # Linear regression (y = mX + c) m is the weight W and c is the bias b
    return X @ W + b

def squared_loss(y_hat, y): # Loss function for regression
    return ((y_hat - y.reshape(y_hat.shape)) ** 2 / 2).mean()

def cross_entropy(y_hat, y): # Loss function for multi-class problem
    n = y.shape[0]  # Number of examples/samples
    loss = -torch.sum(y * torch.log(y_hat)) / n
    return loss

def sgd(params, lr, batch_size): # Optimizer of loss function for non-convex graph
    """ Minibatch stochastic gradient descent """
    # lr = lr / batch_size
    with torch.no_grad(): #  disables gradient calculation in PyTorch, we do not want to calculate gradient during backpropagation
        for param in params:
            param -= lr * param.grad # Manually update parameters (weights and bias) 
            param.grad.zero_() # Reset gradients. new gradients will be calculated during next forward pass

### Training
for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        y_pred = linear(X, w, b)
        loss = squared_loss(y_pred , y)  # Minibatch loss in `X` and `y`
        loss.backward() # Now do backpropagation to calculate  for each batch (very efficient)
        sgd([w, b], lr, batch_size)  # Update parameters using their gradient for each batch (very efficient)
    with torch.no_grad(): # Finally, see the loss after 1 epoch training (make sure calculating gradient is disabled in this step)
        train_loss = squared_loss(linear(features, w, b), labels)
```

### Neural network in pytorch

```
# Creating a neural network
import torch.nn as nn
input_tensor = torch.tensor( [[0.3471, 0.4547, -0.2356]]) # Tensor of shape 1X3 (1 sample of 3 features)
linear_layer = nn.Linear(in_features=3, out_features=2) # Dense layer that takes 3 features and gives output 2 features
nn.init.uniform_(linear_layer.weight) # Avoid exploding gradients due to non-normalized weights
output_tensor = linear_layer(input_tensor) # Passing input tensors to the layer to get ourput tensors (output = W0 @ input + b0)
leaky_relu_layer = nn.LeakyReLU(negative_slope = 0.05)
output_val = leaky_relu_layer(output_tensor)
probabilities_layer = nn.Softmax(dim=-1) # Activation layer applied to the tensor's last dimension (similar to np.argmax(axis=-1))
# You can use nn.Sigmoid() for binary classification, and no activation layer to get regression
output_class = probabilities_layer(output_tensor)
# you can also construct a model like this : model = nn.Sequential(layer0, layer1)
inear_layer.weight # Get the weight of the hidden layer
linear_layer.bias # Get the bias of the hidden layer

# Creating sequential neural network
model = nn.Sequential(
    nn.Linear(6, 4), # First linear layer
    nn.Linear(4, 1), # Second linear layer
    nn.ReLU(), # ReLU activation function as a layer
    nn.Dropout(p=0.5), # 50% probability of neurons to be dropped out, always added after activation functions
    nn.Linear(1, 1), # Third linear layer
    nn.Sigmoid() # Sigmoid activation function as a layer
)
prediction = model(sample)
weight, bias = model.parameters() # extract weight and bias from models all layers

### One-hot the labels 
import torch.nn.functional as F
F.one_hot(torch.tensor(0), num_classes = 3) # tensor([1, 0, 0]), torch.tensor([0]) will make it tensor([[1, 0, 0]])

# Loss function
from torch.nn import CrossEntropyLoss, MSELoss
criterion = CrossEntropyLoss() # Consider this another layer
loss = criterion(prediction, target)
loss.backward()

# How model updates the weights (Demonstrated for only first layer)
lr = 0.001 # Learning rate is typically small
weight = model[0].weight # get weight of first layer
weight_grad = model[0].weight.grad # Get the gradient of first layer weight that was received after backpropagation
weight = weight - lr * weight_grad # Update the weights
bias = model[0].bias # get bias of first layer
bias_grad = model[0].bias.grad # Get the gradient of first layer bias that was received after backpropagation
bias = bias - lr * bias_grad # Update the biases

# Optimization of loss function (to deal with non-convex solution with multiple local minima)
import torch.optim as optim
# Weight decay adds penalty to loss function to discourage large weights and biases
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.95,weight_decay=1e-4) # Momentum for inertia to escape local minimum

# Training
model.train() # Set the model to training mode
from torch.utils.data import TensorDataset, DataLoader
X = df.iloc[:, 1:-1].to_numpy()
y = df.iloc[:, -1].to_numpy()
dataset = TensorDataset(torch.tensor(X).float(), torch.tensor(y).float())
# Access an individual sample
sample = dataset[0]
input_sample, label_sample = sample
dataloader = DataLoader(dataset, batch_size=4, shuffle=True) # partitions data into 4 equal batch data
import torchmetrics
metric = torchmetrics.Accuracy(task="multiclass", num_classes=3)
training_loss = 0.0
for epoch in range(num_epochs): # Train for a number of epochs
    for i, data in enumerate(dataloader, 0):: # train in batches
        optimizer.zero_grad() # Set the gradients to zero
        feature, target = data # Get feature and target from the data loader
        pred = model(feature) # Run a forward pass to get prediction
        loss = criterion(pred, target) # Compute loss and gradients (make sure they are in .float() data type)
        loss.backward() # Run a backward pass (This only calculates the gradients)
        optimizer.step() # Update the parameters by running the optimizer (This is responsible for updating params using gradients)
        training_loss += loss.item()
        acc = metric(targets, predictions.argmax(dim=-1)) # Measuring accuracy of multi-class problem
        acc = metric.compute() # Calculate accuracy over the whole epoch
    metric.reset() # Reset the metric for the next epoch (training or validation)

epoch_loss = training_loss / len(dataloader) # mean loss across all epochs
# Evaluate model
model.eval() # Set the model to evaluation mode
with torch.no_grad():  # No need to track gradients during evaluation
    for i, data in enumerate(validation_dataloader, 0):  # Change to your validation/test dataloader
        # same as training accuracy loss calculation
from torchsummary import summary
summary(model, (3, 32, 32))  # Adjust input size accordingly to see model summary
print(*model.parameters())
for params in model.parameters():
    print(params.shape)
    print(params.numel())
    print(type(params))
# NOTE : also see TRANSFER LEARNING
```

# Chapter 2

<center><img src="images/02.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.02.png"  style="width: 400px, height: 300px;"/></center>

# Chapter 3

- - gradients represent the slope of the loss function with respect to the model parameters of a neural network
- Duying Neuron Problem:
	- This happens when a neuron takes a value less than 0 for all the rows of the data
    - When relu is used and when the slope is 0, 
    	updated weight = current weight - node val * slope
        => updated weight = current weight (no change,  neuron comes to a standstill)
	- Occurs when a neuron keeps getting negative inputs
    - Solution: Use different activation function other than relu (eg: tanh). But tanh will produce vanishing gradient problem
- Vanishing Gradients:
	- Prevents neural networks from booming sooner (when updates to weights 
		during backpropagation are close to 0)
	- Happens when sigmoid or softmax activation function is used
    - For high and low value of x, the gradients approach to 0 (saturation behavior)
	- Occurs when many layers of neural network have very small slopes (e.g. due to
		being on the flat part of the tangent curve
	- Sigmoid function makes the intermediate values of the neural network 
		between 0 and 1 
	- When back-propagation is used, we keep multiplying values less 
		than 1 with each other. 
	- So the gradient keeps getting smaller and smaller moving 
		backward to the network.
	- So, the neurons in the earlier layers learn slowly 
		compared to the neurons in the later layers in the network
	- Thus, training takes too long and accuracy is compromised.
    - Use RELU activation : gradients do not converge to 0 for high value of x
    - Use Leaky RELU activation :For negative inputs, it multiplies the input by a small coefficient (defaulted to 0.01)
    	- For this, gradients for negative inputs are never null for leaky relu
- Exploding Gradients:
	- The opposite of vanishing gradients
	- gradients become so large that it overwhelms the model
	- It happens when the inputs and the weights of a layer are not normalized.

<center><img src="images/03.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.03.png"  style="width: 400px, height: 300px;"/></center>


### Impact of Learning Rate on model

<center><img src="images/03.04.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.05.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.06.png"  style="width: 400px, height: 300px;"/></center>


### Impact of Momentum on model


<center><img src="images/03.07.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.08.png"  style="width: 400px, height: 300px;"/></center>


```

import numpy as np
import matplotlib.pyplot as plt
def optimize_and_plot(lr=0.01, momentum=0.0):
  x = torch.tensor(2.0, requires_grad=True)
  buffer = torch.zeros_like(x.data)
  values = []
  for i in range(10):

      y = function(x)
      values.append((x.clone(), y.clone()))
      y.backward()

      d_p = x.grad.data
      if momentum !=0 :
          buffer.mul_(momentum).add_(d_p)
          d_p = buffer

      x.data.add_(d_p, alpha=-lr)
      x.grad.zero_()
      
  x = np.arange(-3, 2, 0.001)
  y = function(x)

  plt.figure(figsize=(10, 5))
  plt.plot([v[0].detach().numpy() for v in values], [v[1].detach().numpy() for v in values], 'r-X', 
           linewidth=2, markersize=7)
  for i in range(10):
      plt.text(values[i][0]+0.1, values[i][1], f'step {i}', fontdict={'color': 'r'})
  plt.plot(x, y, linewidth=2)
  plt.grid()
  plt.tick_params(axis='both', which='major', labelsize=12)
  plt.legend(['Optimizer steps', 'Square function'])
  plt.show()

def function(x):
    return x**4 + x**3 - 5*x**2
    # return x**6 + x**5 - 5*x**4

# Try a first learning rate value
lr0 = 0.01 # also do this for 0.1, 0.09
optimize_and_plot(lr=lr0)
# Try a first value for momentum
mom0 = 0.3 # also do this for 0.94
optimize_and_plot(momentum=mom0)
```

### Transfer Learning

```
# A new model's initialization of parameters are done with random values
# Transfer learning is nothing but copying a pre-trained model's parameters to a new untrained model as initial parameters 
# It cuts down training time and creates optimum way to better train the model
# Not every layer is trained (we freeze some of them)
# Rule of thumb: freeze early layers of network and fine-tune layers closer to output layer

import torch
layer = nn.Linear(64, 128) 
torch.save(layer, 'layer.pth') # Saving layer with parameters
new_layer = torch.load('layer.pth') # Loading layer with parameters 

model = nn.Sequential(nn.Linear(64, 128), 
                    nn.Linear(128, 256))
for name, param in model.named_parameters():
    if name == '0.weight': # For selected layer
        param.requires_grad = False # Make the parameter constant (Freeze the training)

# Example 2

from collections import OrderedDict

model = nn.Sequential(
    OrderedDict([
        ('layer1', nn.Linear(in_features=8, out_features=16, bias=True)),
        ('layer2', nn.Linear(in_features=16, out_features=32, bias=True)),
        ('layer3', nn.Linear(in_features=32, out_features=10, bias=True))
    ])
)

for name, param in model.named_parameters():
    print(name)

for name, param in model.named_parameters():    
  
    # Check if the parameters belong to the first layer
    if name == 'layer1.weight' or name == 'layer1.bias':
      
        # Freeze the parameters
        param.requires_grad = False
  
    # Check if the parameters belong to the second layer
    if name == 'layer2.weight' or name == 'layer2.bias':
      
        # Freeze the parameters
        param.requires_grad = False

```